## GPT prompting: example notebook with all prompts

### requires python >= 3.10


In [1]:
# for auto-reloading extenrnal modules
# see http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
from tqdm import tqdm
import openai 
import os
from openai import AzureOpenAI
import configparser
import json
import csv
import sys

In [3]:
#sys.path.append("../../../")
sys.path.append("../")
from common_code.gpt_utils import *
from common_code.gpt_reply_formats import *

In [4]:

from prompts.semantic_categories.v01.prompt import (
    ALIVE_SYSTEM_PROMPT, ALIVE_FEW_SHOTS_STR, ALIVE_FEW_SHOTS,
    EVENT_SYSTEM_PROMPT, EVENT_FEW_SHOTS_STR, EVENT_FEW_SHOTS,
    LOC_SYSTEM_PROMPT, LOC_FEW_SHOTS_STR, LOC_FEW_SHOTS,
    ABSTRACT_SYSTEM_PROMPT, ABSTRACT_FEW_SHOTS_STR, ABSTRACT_FEW_SHOTS,
    TIME_SYSTEM_PROMPT, TIME_FEW_SHOTS_STR, TIME_FEW_SHOTS,
    STATE_SYSTEM_PROMPT, STATE_FEW_SHOTS_STR, STATE_FEW_SHOTS,
)

In [5]:
pd.set_option('display.max_colwidth', None)
pd.set_option("display.show_dimensions", True)

In [8]:
INPUT_FILE = "../data/ELT_n50_500_sample.csv"

OUTPUT_FILE = "../results/ELT_n50_500_sample_tagged_new_order.csv"

AI_CONF_FILE = "../../../v04_verb-case_pattern/minu_code/azure.ini"

BS = 10

# GPT

## GPT jaoks vajalik

In [9]:
config = configparser.ConfigParser()

status = config.read(AI_CONF_FILE) 
assert status == [AI_CONF_FILE]

API_VERSION = config['azure-configuration']['api_version']
AZURE_ENDPOINT = config['azure-configuration']['api_base']
SUBSCRIPTION_KEY = config['azure-configuration']['api_key']
model_name = "gpt-4o" #"GPT-4o-2024-1120 Global"
DEPLOYMENT = config['azure-configuration']['deployment_id']

In [10]:
client = AzureOpenAI(
    api_version=API_VERSION,
    azure_endpoint=AZURE_ENDPOINT,
    api_key=SUBSCRIPTION_KEY,
)

## Functions for classifying and explanation

In [11]:

def classify_batch(my_batch, few_shots, system_prompt, client, deployment):
    """Gets a yes/no answer for a batch of sentences and phrases. 
    """
    #print("classify", len(my_batch))
    max_att = 1
    attempt = 0
    while attempt < max_att:
        attempt += 1
        user_payload = {
            "few_shots": few_shots,
            "batch": my_batch
        }
    
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content":  json.dumps(user_payload, ensure_ascii=False)}
        ]

        #return None, None
        response = client.chat.completions.create(
            model=deployment,
            messages=messages,
            temperature=0, # absoluutselt min väljund 
        )

        raw_output = response.choices[0].message.content.strip()

        try:
            data = json.loads(raw_output)

            if len(data) != len(my_batch):
                raise ValueError(f"Väljundis ei ole õige arv vastuseid. Peaks olema {len(batch)} aga on {len(data)}.")
                
            elif len(data) == len(my_batch):
                for item in data:
                    ClassificationDict(**item)

            return response, raw_output

        except (ValidationError, json.JSONDecodeError, ValueError) as e:
            #print(f"Attempt {attempt} failed. Retrying batch...")
            print(f"Error: {e}")
            #print(f"Raw output: {raw_output[:500]}...")  # preview first 500 chars
            time.sleep(1)  # small delay before retry

    print(f"Batch failed after {max_att} attempts.")
    # isegi kui ei saanud kõike kätte siis saab pärast äkki käsitsi midagi juurde panna
    return response, raw_output



In [14]:
def get_results(df, bs, few_shots, system_prompt, client, deployment, max_allowed_tok=None ):
    
    """Makes the gpt query and returns results with how many tokens were used."""
    
    results = []
    results2 = []
    responses = []

    used_tokens = 0
    batch_cnt = 0

    rows = df.to_dict(orient="records")
    # kui tahta kõiki näiteid anda gpt-le
    for df_batch in tqdm(chunk_data(rows, size=bs)):
        
        batch = []
        for ex in df_batch:
            batch.append( json.dumps({"l": ex["sentence"], "c": ex["form"]}, ensure_ascii=False))

        batch_cnt += 1
        # klassifitseeri
        result_yesno = []
        response, result = classify_batch(batch, few_shots, system_prompt, client, deployment)
        result_yesno = json.loads(result)
        results += result_yesno
        results2.append(result_yesno)
        responses.append(response)
        used_tokens += response.usage.total_tokens

        if max_allowed_tok is not None and used_tokens >= max_allowed_tok:
            print(f"Tehtud on {batch_cnt} batchi ehk {batch_cnt*bs} lauset")
            break    
            
    return results, used_tokens

In [15]:
def get_rows(df, decision_columns):
    """Get data rows for gpt. Based on columns that have to be None/'no'."""
    mask = pd.Series(True, index=df.index)

    for col in decision_columns:
        if col in ["A", "S", "T", "L", "L1", "L2", "E"]:
            mask &= df[col] == "no"
        elif col in ["ner_tag", "timex_tag"]:
            mask &= df[col].isna()

    filtered = df[mask]

    return filtered

In [16]:
def merge_data(df1, df, tag):
    key_cols = ['sentence_id','head_id', "verb", "verb_compound", "morph_case", "form"]

    df_selected = df[key_cols + [tag]].copy()

    df1['verb_compound'] = df1['verb_compound'].astype('string').str.strip()
    df_selected['verb_compound'] = df_selected['verb_compound'].astype('string').str.strip()

    # Merge df1 with df2_selected etc
    merged_df = df1.merge(df_selected, on=key_cols, how='left')

    merged_df[tag] = merged_df[tag].fillna("no")
    
    return merged_df
    

# ALIVE -> EVENT -> TIME -> LOC -> STATE

# ALIVE

In [17]:
df1 = pd.read_csv(INPUT_FILE, encoding="utf-8",  sep=",")

FILTER_COLS = "ner_tag,timex_tag".split(",") 
df2 = get_rows(df1, FILTER_COLS)

df = df2.sample(frac=1).copy()

tag = "A"

In [19]:
results, used_tokens = get_results(df, BS, ALIVE_FEW_SHOTS_STR, ALIVE_SYSTEM_PROMPT, client, DEPLOYMENT)

9it [00:07,  1.23it/s]


In [20]:
used_tokens

9905

In [22]:
# andmed tabelisse ja faili

assert len(results) == len(df)

if len(results) == len(df):
    df[tag] = [r["a"] for r in results]
    
merged_df = merge_data(df1, df, tag)

In [23]:
merged_df.to_csv(OUTPUT_FILE, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

# EVENT

In [25]:
df1 = pd.read_csv(OUTPUT_FILE, encoding="utf-8",  sep=",")

FILTER_COLS = "ner_tag,timex_tag,A".split(",") 
df2 = get_rows(df1, FILTER_COLS)
df = df2.sample(frac=1).copy()

tag = "E"

In [27]:
results, used_tokens = get_results(df, BS, EVENT_FEW_SHOTS_STR, EVENT_SYSTEM_PROMPT, client, DEPLOYMENT)

7it [00:05,  1.36it/s]


In [28]:
used_tokens

10165

In [29]:
# andmed tabelisse ja faili

assert len(results) == len(df)

if len(results) == len(df):
    df[tag] = [r["a"] for r in results]
        
merged_df = merge_data(df1, df, tag)

In [30]:
merged_df.to_csv(OUTPUT_FILE, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

# TIME

In [32]:
df1 = pd.read_csv(OUTPUT_FILE, encoding="utf-8",  sep=",")

FILTER_COLS = "ner_tag,timex_tag,A,E".split(",") 
df2 = get_rows(df1, FILTER_COLS)

df = df2.sample(frac=1).copy()

tag = "T"

In [34]:
results, used_tokens = get_results(df, BS, TIME_FEW_SHOTS_STR, TIME_SYSTEM_PROMPT, client, DEPLOYMENT)

6it [00:05,  1.09it/s]


In [35]:
used_tokens 

6108

In [39]:
# andmed tabelisse ja faili

assert len(results) == len(df)

if len(results) == len(df):
    df[tag] = [r["a"] for r in results]
        
merged_df = merge_data(df1, df, tag)

In [40]:
merged_df.to_csv(OUTPUT_FILE, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

# LOCATION

In [41]:
df1 = pd.read_csv(OUTPUT_FILE, encoding="utf-8",  sep=",")

FILTER_COLS = "ner_tag,timex_tag,A,E,T".split(",") 
df2 = get_rows(df1, FILTER_COLS)

df = df2.sample(frac=1).copy()

tag = "L1"

In [43]:
results, used_tokens = get_results(df, BS, LOC_FEW_SHOTS_STR, LOC_SYSTEM_PROMPT, client, DEPLOYMENT)

6it [00:04,  1.29it/s]


In [44]:
used_tokens

19608

In [45]:
# andmed tabelisse ja faili

assert len(results) == len(df)

if len(results) == len(df):
    df[tag] = [r["a"] for r in results]
        
merged_df = merge_data(df1, df, tag)

In [46]:
merged_df.to_csv(OUTPUT_FILE, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

## Abstract location

In [49]:
df1 = pd.read_csv(OUTPUT_FILE, encoding="utf-8",  sep=",")

FILTER_COLS = "ner_tag,timex_tag,A,E,T,L1".split(",") 
df2 = get_rows(df1, FILTER_COLS)

df = df2.sample(frac=1).copy()

tag = "L2"

In [51]:
results, used_tokens = get_results(df, BS, ABSTRACT_FEW_SHOTS_STR, ABSTRACT_SYSTEM_PROMPT, client, DEPLOYMENT)

3it [00:02,  1.38it/s]


In [52]:
assert len(results) == len(df)

if len(results) == len(df):
    df[tag] = [r["a"] for r in results]
        
merged_df = merge_data(df1, df, tag)

In [53]:
merged_df.to_csv(OUTPUT_FILE, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

In [54]:
### merge L1 (location) and L2 (abstract location) into one column "L"

df1 = pd.read_csv(OUTPUT_FILE, encoding="utf-8",  sep=",")

df1['L'] = ((df1['L1'] == 'yes') | (df1['L2'] == 'yes')).map({True: 'yes', False: 'no'})
#df1['L'] = df1[['L1', 'L2']].eq('yes').any(axis=1).map({True: 'yes', False: 'no'})

df1.to_csv(OUTPUT_FILE, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

# STATE

In [55]:
df1 = pd.read_csv(OUTPUT_FILE, encoding="utf-8",  sep=",")

FILTER_COLS = "ner_tag,timex_tag,A,E,T,L".split(",") 
df2 = get_rows(df1, FILTER_COLS)

df = df2.sample(frac=1).copy()

tag = "S"

In [57]:
results, used_tokens = get_results(df, BS, STATE_FEW_SHOTS_STR, STATE_SYSTEM_PROMPT, client, DEPLOYMENT)

2it [00:01,  1.15it/s]


In [58]:
assert len(results) == len(df)

if len(results) == len(df):
    df[tag] = [r["a"] for r in results]
        
merged_df = merge_data(df1, df, tag)

In [59]:
merged_df.to_csv(OUTPUT_FILE, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

# Create new final tag column

In [60]:
# uus classification2
# muuta vastavalt vajadusele

def new_class(row):
    mapping = {"alive":"A", "PER": "A", "location":"L", "LOC":"L", "time":"T", "event":"E", "state":"S"}

    if str(row["ner_tag"]) != "" and str(row["ner_tag"]) != "nan":
        if row["ner_tag"] == "ORG":
            if row["morph_case"] in ["adit", "in", "ill", "el"]: #sisekohakäänetes
                return "L"
            else: # väliskohakäänetes
                return "A"
        else:
            return mapping[row["ner_tag"]]
    
    if str(row["timex_tag"]) != "" and str(row["timex_tag"]) != "nan":
        return "T"

    if row["L"] == "yes":
        return "L"
    
    # kui on aeg -> "yes" -> saame välja visata
    if row["T"] == "yes":
        return "T"
    
    # event -> "yes" -> saame välja visata
    if row["E"] == "yes":
        return "E"
    
    # elus -> "yes" -> saame välja visata
    if row["A"] == "yes":
        return "A"

    if row["S"] == "yes":
        return "S"
    
    
    # kui oli "no", NaN ja/või alive/time/abstract kõik olid "no"
    else:
        return ""

In [61]:
df = pd.read_csv(OUTPUT_FILE, encoding="utf-8",  sep=",")

In [62]:
df["tag"] = df.apply(new_class, axis=1)

In [63]:
df.to_csv(OUTPUT_FILE, encoding="utf-8", index=False, sep=",", quoting=csv.QUOTE_MINIMAL)